In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Fabric Metadata & Data Quality Assessment Framework\n",
    "\n",
    "## 1. Overview & Architecture\n",
    "This framework provides an automated, end-to-end data quality assessment for Microsoft Fabric Lakehouses. It extracts structural metadata, profiles columns, evaluates custom quality rules, generates an aggregate quality index, and outputs an actionable summary report.\n",
    "\n",
    "### Platform Placement\n",
    "```\n",
    "             Microsoft Fabric\n",
    "                    │\n",
    "                 OneLake\n",
    "                    │\n",
    "                 Lakehouse\n",
    "                    │\n",
    "             ┌──────┴──────┐\n",
    "             │             │\n",
    "          Metadata      Dataset\n",
    "             │             │\n",
    "             └──────┬──────┘\n",
    "                    │\n",
    "             Data Profiling\n",
    "                    │\n",
    "          Data Quality Rules\n",
    "                    │\n",
    "             Quality Score\n",
    "                    │\n",
    "             Assessment Report\n",
    "```"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Configuration & Mode Selection\n",
    "Set `USE_SYNTHETIC_DATA = True` to run the demonstration mode without external dependencies. Set to `False` and configure `TARGET_TABLE` to profile an existing Delta table in your Fabric Lakehouse."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Configuration Parameters\n",
    "USE_SYNTHETIC_DATA = True\n",
    "TARGET_TABLE = \"default_lakehouse_table\"  # Used if USE_SYNTHETIC_DATA is False\n",
    "SYNTHETIC_ROW_COUNT = 10000\n",
    "\n",
    "print(f\"Execution Mode: {'Synthetic Data Generation' if USE_SYNTHETIC_DATA else f'Lakehouse Table ({TARGET_TABLE})'}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Data Ingestion / Synthetic Data Generation\n",
    "Generates a synthetic e-commerce retail dataset containing artificial data quality anomalies (nulls, duplicates, invalid quantities, negative prices) to demonstrate framework capabilities."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from pyspark.sql import SparkSession\n",
    "from pyspark.sql.functions import col, count, when, isnull, sum as _sum, avg, min as _min, max as _max, lit, rand, expr\n",
    "from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType\n",
    "import datetime\n",
    "\n",
    "spark = SparkSession.builder.getOrCreate()\n",
    "\n",
    "if USE_SYNTHETIC_DATA:\n",
    "    # Generate base dataset with intentional data quality issues\n",
    "    df_base = spark.range(0, SYNTHETIC_ROW_COUNT).select(\n",
    "        col(\"id\").cast(\"integer\").alias(\"order_id\"),\n",
    "        # 0.2% null customer IDs\n",
    "        when(rand() < 0.002, lit(None)).otherwise((col(\"id\") % 1000 + 100).cast(\"string\")).alias(\"customer_id\"),\n",
    "        # 1.4% duplicate records\n",
    "        when(rand() < 0.014, lit(101)).otherwise((col(\"id\") % 500 + 1).cast(\"integer\")).alias(\"product_id\"),\n",
    "        # 0.0% invalid quantities (all >= 1)\n",
    "        (expr(\"abs(cast(rand() * 5 as int)) + 1\")).alias(\"quantity\"),\n",
    "        # 0.3% invalid sales amounts (negative prices)\n",
    "        when(rand() < 0.003, -10.0).otherwise(expr(\"round(rand() * 100 + 10, 2)\")).alias(\"unit_price\"),\n",
    "        # Timestamp\n",
    "        expr(\"current_timestamp()\").alias(\"transaction_timestamp\")\n",
    "    )\n",
    "    df_target = df_base\n",
    "    table_name = \"demo_synthetic_retail\"\n",
    "else:\n",
    "    df_target = spark.table(TARGET_TABLE)\n",
    "    table_name = TARGET_TABLE\n",
    "\n",
    "df_target.cache()\n",
    "total_rows = df_target.count()\n",
    "print(f\"Target Dataset '{table_name}' loaded successfully. Total Rows: {total_rows:,}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Metadata Discovery\n",
    "Extracts schema specifications, column physical data types, nullability properties, and table-level dimensions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "metadata_schema = []\n",
    "for field in df_target.schema.fields:\n",
    "    metadata_schema.append({\n",
    "        \"Column Name\": field.name,\n",
    "        \"Data Type\": field.dataType.simpleString(),\n",
    "        \"Nullable\": field.nullable\n",
    "    })\n",
    "\n",
    "df_metadata = spark.createDataFrame(metadata_schema)\n",
    "print(f\"=== METADATA PROFILE: {table_name} ===\")\n",
    "df_metadata.show(truncate=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Statistical Data Profiling\n",
    "Computes statistical profiles for every attribute, including null ratios, distinct cardinality, min/max values, and duplicate frequency."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "profiling_results = []\n",
    "\n",
    "for column_name in df_target.columns:\n",
    "    null_cnt = df_target.filter(col(column_name).isNull() | isnull(col(column_name))).count()\n",
    "    distinct_cnt = df_target.select(column_name).distinct().count()\n",
    "    min_val = str(df_target.select(_min(col(column_name))).collect()[0][0])\n",
    "    max_val = str(df_target.select(_max(col(column_name))).collect()[0][0])\n",
    "    \n",
    "    profiling_results.append({\n",
    "        \"Column\": column_name,\n",
    "        \"Null Count\": null_cnt,\n",
    "        \"Null Percentage\": round((null_cnt / total_rows) * 100, 2),\n",
    "        \"Distinct Values\": distinct_cnt,\n",
    "        \"Min Value\": min_val,\n",
    "        \"Max Value\": max_val\n",
    "    })\n",
    "\n",
    "df_profile = spark.createDataFrame(profiling_results)\n",
    "print(\"=== DATA PROFILING SUMMARY ===\")\n",
    "df_profile.show(truncate=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Data Quality Rules Execution\n",
    "Applies logical data quality validations (Completeness, Uniqueness, Validity) across target columns."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dq_checks = []\n",
    "\n",
    "# Check 1: Completeness - Customer ID Nulls\n",
    "null_cust_cnt = df_target.filter(col(\"customer_id\").isNull()).count()\n",
    "null_cust_pct = round((null_cust_cnt / total_rows) * 100, 2)\n",
    "dq_checks.append({\n",
    "    \"CheckName\": \"Null customer IDs\",\n",
    "    \"Dimension\": \"Completeness\",\n",
    "    \"MetricValue\": f\"{null_cust_pct}%\",\n",
    "    \"Status\": \"PASS\" if null_cust_pct <= 0.5 else \"FAIL\",\n",
    "    \"Weight\": 20,\n",
    "    \"Passed\": 1 if null_cust_pct <= 0.5 else 0\n",
    "})\n",
    "\n",
    "# Check 2: Uniqueness - Order ID Duplicates\n",
    "distinct_orders = df_target.select(\"order_id\").distinct().count()\n",
    "dup_order_pct = round(((total_rows - distinct_orders) / total_rows) * 100, 2)\n",
    "dq_checks.append({\n",
    "    \"CheckName\": \"Duplicate orders\",\n",
    "    \"Dimension\": \"Uniqueness\",\n",
    "    \"MetricValue\": f\"{dup_order_pct}%\",\n",
    "    \"Status\": \"WARNING\" if 0.5 < dup_order_pct <= 2.0 else (\"PASS\" if dup_order_pct <= 0.5 else \"FAIL\"),\n",
    "    \"Weight\": 25,\n",
    "    \"Passed\": 0.75 if 0.5 < dup_order_pct <= 2.0 else (1 if dup_order_pct <= 0.5 else 0)\n",
    "})\n",
    "\n",
    "# Check 3: Validity - Quantity > 0\n",
    "invalid_qty_cnt = df_target.filter(col(\"quantity\") <= 0).count()\n",
    "invalid_qty_pct = round((invalid_qty_cnt / total_rows) * 100, 2)\n",
    "dq_checks.append({\n",
    "    \"CheckName\": \"Invalid quantities\",\n",
    "    \"Dimension\": \"Validity\",\n",
    "    \"MetricValue\": f\"{invalid_qty_pct}%\",\n",
    "    \"Status\": \"PASS\" if invalid_qty_pct == 0.0 else \"FAIL\",\n",
    "    \"Weight\": 25,\n",
    "    \"Passed\": 1 if invalid_qty_pct == 0.0 else 0\n",
    "})\n",
    "\n",
    "# Check 4: Validity - Unit Price > 0\n",
    "invalid_price_cnt = df_target.filter(col(\"unit_price\") <= 0).count()\n",
    "invalid_price_pct = round((invalid_price_cnt / total_rows) * 100, 2)\n",
    "dq_checks.append({\n",
    "    \"CheckName\": \"Invalid sales amount\",\n",
    "    \"Dimension\": \"Validity\",\n",
    "    \"MetricValue\": f\"{invalid_price_pct}%\",\n",
    "    \"Status\": \"PASS\" if invalid_price_pct <= 0.5 else \"FAIL\",\n",
    "    \"Weight\": 30,\n",
    "    \"Passed\": 1 if invalid_price_pct <= 0.5 else 0\n",
    "})\n",
    "\n",
    "df_dq_results = spark.createDataFrame(dq_checks)\n",
    "df_dq_results.select(\"CheckName\", \"Dimension\", \"MetricValue\", \"Status\").show(truncate=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. Data Quality Index (DQI) Calculation\n",
    "Computes a weighted total percentage index evaluating overall dataset health."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "total_weight = sum([r[\"Weight\"] for r in dq_checks])\n",
    "weighted_passed = sum([r[\"Weight\"] * r[\"Passed\"] for r in dq_checks])\n",
    "overall_dq_score = round((weighted_passed / total_weight) * 100, 1)\n",
    "\n",
    "print(\"=\" * 45)\n",
    "print(f\"  OVERALL DATA QUALITY SCORE: {overall_dq_score}%\")\n",
    "print(\"=\" * 45)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Final Assessment Report\n",
    "Formats an executive scorecard table showing individual validation metrics."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"+-----------------------+--------------+--------+\")\n",
    "print(\"| CheckResult           | Status       | Metric |\")\n",
    "print(\"+-----------------------+--------------+--------+\")\n",
    "print(f\"| Row count             | PASS         | {total_rows:,}\")\n",
    "for r in dq_checks:\n",
    "    check = r['CheckName'].ljust(21)\n",
    "    status = r['Status'].ljust(12)\n",
    "    metric = r['MetricValue'].ljust(6)\n",
    "    print(f\"| {check} | {status} | {metric} |\")\n",
    "print(\"+-----------------------+--------------+--------+\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Automated Recommendations & Remediation\n",
    "Provides automated next steps based on the validation rule output."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=== AUTOMATED REMEDIATION RECOMMENDATIONS ===\")\n",
    "for r in dq_checks:\n",
    "    if r[\"Status\"] == \"WARNING\":\n",
    "        print(f\"- [WARNING] {r['CheckName']}: Metric is {r['MetricValue']}. Apply `dropDuplicates(['order_id'])` transformation downstream.\")\n",
    "    elif r[\"Status\"] == \"FAIL\":\n",
    "        print(f\"- [ACTION REQUIRED] {r['CheckName']}: Metric is {r['MetricValue']}. Investigate source pipeline for bad input data.\")\n",
    "\n",
    "if overall_dq_score >= 85.0:\n",
    "    print(\"\\nDataset Status: APPROVED for Gold Layer / Downstream Analytics Consumption.\")\n",
    "else:\n",
    "    print(\"\\nDataset Status: REJECTED for Production Use. Data remediation required.\")"
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}